In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Transpile & Evaluate C Pipeline Performance Dropoff (`models/transpile_soft_pipeline_to_c.ipynb`)

This notebook transpiles the **OOF Multinomial Logistic Regression Meta-Learner** to pure C code, compiles it into a shared library (`deploy/libtriage_pipeline.so`), runs inference on the 1% Holdout Test Set via `ctypes`, and **quantifies the exact performance dropoff between Python/R Native vs. Embedded C implementation** across all ESI classes:

### Evaluation Metrics Tracked
1. **Maximum Probability Difference**: $\max |P_{\text{C}} - P_{\text{Python}}|$
2. **Prediction Mismatch Rate**: % of test samples where $\hat{y}_{\text{C}} \neq \hat{y}_{\text{Python}}$
3. **Per-Class Metrics**: Recall, Specificity, Balanced Accuracy, ROC-AUC for both Python & C

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load deploy/oof_multinomial_logistic_meta_learner.rds & Export Model Artifacts
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(lightgbm)
})
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
rds_path <- file.path(deploy_dir, "oof_multinomial_logistic_meta_learner.rds")
if (!file.exists(rds_path)) {
  stop(paste("Error:", rds_path, "does not exist! Please run models/train_oof_logistic_regression_stacking.ipynb first."))
}
bundle <- readRDS(rds_path)
# Save LightGBM Boosters as Model Text Files for Python
lgb.save(bundle$l1_model,  file.path(deploy_dir, "l1_booster.txt"))
saveRDS(bundle$l1_model,   file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
lgb.save(bundle$l2_model,  file.path(deploy_dir, "l2_booster.txt"))
saveRDS(bundle$l2_model,   file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
lgb.save(bundle$l3a_model, file.path(deploy_dir, "l3a_booster.txt"))
saveRDS(bundle$l3a_model,  file.path(deploy_dir, "lightgbm_esi23_model.rds"))
lgb.save(bundle$l3b_model, file.path(deploy_dir, "l3b_booster.txt"))
saveRDS(bundle$l3b_model,  file.path(deploy_dir, "lightgbm_esi45_model.rds"))
# Export Meta-Learner Parameters and Scaler as JSON
meta_info <- list(
  intercepts  = as.numeric(bundle$intercepts),
  coef_matrix = lapply(1:nrow(bundle$coef_matrix), function(i) as.numeric(bundle$coef_matrix[i, ])),
  scaler      = list(means = as.list(bundle$scaler$means), sds = as.list(bundle$scaler$sds), cols = bundle$scaler$cols),
  feature_cols= bundle$feature_cols
)
write(jsonlite::toJSON(meta_info, auto_unbox = TRUE), file.path(deploy_dir, "meta_learner_info.json"))
cat("Successfully loaded deploy/oof_multinomial_logistic_meta_learner.rds and exported booster & JSON artifacts!\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Direct AST Tree Transpilation & Meta-Learner m2cgen Export
# ---------------------------------------------------------
import os
import sys
import json
import numpy as np
try:
    import m2cgen as m2c
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "m2cgen"])
    import m2cgen as m2c
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
with open(os.path.join(deploy_dir, 'meta_learner_info.json')) as f:
    meta_info = json.load(f)
intercepts  = np.array(meta_info['intercepts'], dtype=np.float64)
coef_matrix = np.array(meta_info['coef_matrix'], dtype=np.float64)
scaler_means = meta_info['scaler']['means']
scaler_sds   = meta_info['scaler']['sds']
scaler_cols  = meta_info['scaler']['cols']
feature_names = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif'
]
def transpile_lgb_node(node):
    if 'leaf_value' in node:
        val = float(node['leaf_value'])
        return f"return {val:.8f}f;"
    
    feat_idx = int(node['split_feature'])
    thresh   = float(node['threshold'])
    left_c   = transpile_lgb_node(node['left_child'])
    right_c  = transpile_lgb_node(node['right_child'])
    
    return f"if (x[{feat_idx}] <= {thresh:.8f}f) {{ {left_c} }} else {{ {right_c} }}"
def transpile_lgb_booster_to_c(booster_file, func_name):
    booster = lgb.Booster(model_file=booster_file)
    dump = booster.dump_model()
    code = f"static float {func_name}(const double x[38]) {{\n"
    code += "    double margin = 0.0;\n"
    for tree in dump['tree_info']:
        tree_c = transpile_lgb_node(tree['tree_structure'])
        code += f"    margin += (double)({{ {tree_c} }});\n"
    code += "    return (float)(1.0 / (1.0 + exp(-margin)));\n}"
    return code
c_code_l1  = transpile_lgb_booster_to_c(os.path.join(deploy_dir, 'l1_booster.txt'),  'predict_layer1')
c_code_l2  = transpile_lgb_booster_to_c(os.path.join(deploy_dir, 'l2_booster.txt'),  'predict_layer2')
c_code_l3a = transpile_lgb_booster_to_c(os.path.join(deploy_dir, 'l3a_booster.txt'), 'predict_layer3a')
c_code_l3b = transpile_lgb_booster_to_c(os.path.join(deploy_dir, 'l3b_booster.txt'), 'predict_layer3b')
meta_logreg = LogisticRegression(multi_class='multinomial', class_weight='balanced')
meta_logreg.classes_ = np.array([1, 2, 3, 4, 5])
meta_logreg.intercept_ = intercepts
meta_logreg.coef_ = coef_matrix
meta_logreg.n_features_in_ = 5
c_code_meta = m2c.export_to_c(meta_logreg, function_name='predict_meta_logistic')
print("Transpiled Decision Trees and Meta-Learner Code Generated!")

In [ ]:
# ---------------------------------------------------------
# Step 3: Assemble Pure C Files & Compile Shared Library (libtriage_pipeline.so)
# ---------------------------------------------------------
import subprocess
header_content = """#ifndef TRIAGE_PIPELINE_H
#define TRIAGE_PIPELINE_H
#ifdef __cplusplus
extern "C" {
#endif
typedef struct {
    float age;
    float cc_breathingdifficulty;
    float gender;
    float triage_vital_hr;
    float triage_vital_sbp;
    float triage_vital_rr;
    float triage_vital_o2;
    float pulse_min;
    float resp_min;
    float spo2_min;
    float sbp_min;
    float pulse_max;
    float resp_max;
    float spo2_max;
    float sbp_max;
} TriageInput;
typedef struct {
    float probs[5];
    int predicted_esi;
} TriageOutput;
TriageOutput predict_triage(const TriageInput* input);
#ifdef __cplusplus
}
#endif
#endif // TRIAGE_PIPELINE_H
"""
scaling_lines = []
for idx, f_name in enumerate(feature_names):
    if f_name in scaler_cols:
        m = scaler_means[f_name]
        s = scaler_sds[f_name]
        scaling_lines.append(f"    x[{idx}] = (x[{idx}] - {m:.8f}) / {s:.8f};")
scaling_c_str = "\n".join(scaling_lines)
c_source_content = f"""#include <math.h>
#include "triage_pipeline.h"
{c_code_l1}
{c_code_l2}
{c_code_l3a}
{c_code_l3b}
{c_code_meta}
TriageOutput predict_triage(const TriageInput* in) {{
    TriageOutput out;
    double x[38];
    
    x[0]  = (double)in->age;
    x[1]  = (double)in->cc_breathingdifficulty;
    x[2]  = (double)in->gender;
    x[3]  = (double)in->triage_vital_hr;
    x[4]  = (double)in->triage_vital_sbp;
    x[5]  = (double)in->triage_vital_rr;
    x[6]  = (double)in->triage_vital_o2;
    x[7]  = (double)in->pulse_min;
    x[8]  = (double)in->resp_min;
    x[9]  = (double)in->spo2_min;
    x[10] = (double)in->sbp_min;
    x[11] = (double)in->pulse_max;
    x[12] = (double)in->resp_max;
    x[13] = (double)in->spo2_max;
    x[14] = (double)in->sbp_max;
    
    x[15] = (in->triage_vital_o2 < 90.0f) ? 1.0 : 0.0;
    x[16] = (in->triage_vital_o2 > 90.0f && in->triage_vital_o2 < 94.0f) ? 1.0 : 0.0;
    x[17] = (in->triage_vital_rr < 10.0f) ? 1.0 : 0.0;
    x[18] = (in->triage_vital_rr > 30.0f) ? 1.0 : 0.0;
    x[19] = (in->triage_vital_sbp <= 90.0f) ? 1.0 : 0.0;
    x[20] = (in->triage_vital_sbp > 220.0f) ? 1.0 : 0.0;
    x[21] = (in->triage_vital_hr < 40.0f) ? 1.0 : 0.0;
    x[22] = (in->triage_vital_hr > 40.0f && in->triage_vital_hr < 60.0f) ? 1.0 : 0.0;
    x[23] = (in->triage_vital_hr > 150.0f) ? 1.0 : 0.0;
    x[24] = (in->triage_vital_hr > 100.0f && in->triage_vital_hr < 150.0f) ? 1.0 : 0.0;
    x[25] = (double)(in->pulse_max - in->pulse_min);
    x[26] = (double)(in->resp_max - in->resp_min);
    x[27] = (double)(in->spo2_max - in->spo2_min);
    x[28] = (double)(in->sbp_max - in->sbp_min);
    
    x[29] = (double)(in->triage_vital_hr / ((in->triage_vital_sbp == 0.0f) ? 1.0f : in->triage_vital_sbp));
    x[30] = (double)(in->triage_vital_hr - x[25]);
    x[31] = (double)(in->triage_vital_sbp - x[28]);
    x[32] = (double)(in->triage_vital_rr - x[26]);
    x[33] = (double)(in->triage_vital_o2 - x[27]);
    x[34] = (double)(in->triage_vital_o2 / ((in->triage_vital_rr == 0.0f) ? 1.0f : in->triage_vital_rr));
    x[35] = (double)(x[27] / ((in->spo2_max == 0.0f) ? 1.0f : in->spo2_max));
    x[36] = (double)(x[25] / (in->triage_vital_hr + 1.0f));
    x[37] = (double)((in->triage_vital_rr / ((in->triage_vital_o2 == 0.0f) ? 1.0f : in->triage_vital_o2)) * 100.0f);
    
{scaling_c_str}
    
    double p1  = (double)predict_layer1(x);
    double p2  = (double)predict_layer2(x);
    double p3a = (double)predict_layer3a(x);
    double p3b = (double)predict_layer3b(x);
    
    double base_probs[5];
    base_probs[0] = p1;
    base_probs[1] = (1.0 - p1) * p2 * p3a;
    base_probs[2] = (1.0 - p1) * p2 * (1.0 - p3a);
    base_probs[3] = (1.0 - p1) * (1.0 - p2) * p3b;
    base_probs[4] = (1.0 - p1) * (1.0 - p2) * (1.0 - p3b);
    
    double meta_probs[5];
    predict_meta_logistic(base_probs, meta_probs);
    
    int best_esi = 1;
    double max_p = meta_probs[0];
    out.probs[0] = (float)meta_probs[0];
    
    for (int k = 1; k < 5; k++) {{
        out.probs[k] = (float)meta_probs[k];
        if (meta_probs[k] > max_p) {{
            max_p = meta_probs[k];
            best_esi = k + 1;
        }}
    }}
    out.predicted_esi = best_esi;
    return out;
}}
"""
h_path  = os.path.join(deploy_dir, 'triage_pipeline.h')
c_path  = os.path.join(deploy_dir, 'triage_pipeline.c')
so_path = os.path.join(deploy_dir, 'libtriage_pipeline.so')
with open(h_path, 'w') as f: f.write(header_content)
with open(c_path, 'w') as f: f.write(c_source_content)
# Compile Shared C Library
compile_cmd = f"gcc -O3 -shared -fPIC -I{deploy_dir} {c_path} -o {so_path} -lm"
res = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if res.returncode != 0:
    print("GCC Compilation Error:", res.stderr)
else:
    print(f"Compiled Shared Library: {so_path}")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Extract Holdout Test Set Inputs in R & Run Python/C Comparison
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(as.character(raw_df[[config$classes$target_col]]), levels = c("1", "2", "3", "4", "5"))
)
df_master <- na.omit(df_master)
test_size <- config$training$test_size
in_train  <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
test_raw_df <- df_master[-in_train, ]
y_test_vec <- as.numeric(as.character(test_raw_df$target_col))
test_inputs_matrix <- as.matrix(test_raw_df[, 1:15])
cat(sprintf("Extracted %d Holdout Test Samples for C Inference\n", nrow(test_inputs_matrix)))

In [ ]:
# ---------------------------------------------------------
# Step 5: Quantify Exact Dropoff (ctypes Shared Library Inference vs Native Python/R)
# ---------------------------------------------------------
import ctypes
from rpy2.robjects import r
test_inputs = np.array(r('test_inputs_matrix'), dtype=np.float32)
y_test      = np.array(r('y_test_vec'), dtype=int)
so_path = os.path.join(deploy_dir, 'libtriage_pipeline.so')
c_lib   = ctypes.CDLL(so_path)
class TriageInput(ctypes.Structure):
    _fields_ = [
        ('age', ctypes.c_float),
        ('cc_breathingdifficulty', ctypes.c_float),
        ('gender', ctypes.c_float),
        ('triage_vital_hr', ctypes.c_float),
        ('triage_vital_sbp', ctypes.c_float),
        ('triage_vital_rr', ctypes.c_float),
        ('triage_vital_o2', ctypes.c_float),
        ('pulse_min', ctypes.c_float),
        ('resp_min', ctypes.c_float),
        ('spo2_min', ctypes.c_float),
        ('sbp_min', ctypes.c_float),
        ('pulse_max', ctypes.c_float),
        ('resp_max', ctypes.c_float),
        ('spo2_max', ctypes.c_float),
        ('sbp_max', ctypes.c_float)
    ]
class TriageOutput(ctypes.Structure):
    _fields_ = [
        ('probs', ctypes.c_float * 5),
        ('predicted_esi', ctypes.c_int)
    ]
c_lib.predict_triage.argtypes = [ctypes.POINTER(TriageInput)]
c_lib.predict_triage.restype  = TriageOutput
# Execute C Shared Library Predictions
c_preds = []
c_probs = []
for row in test_inputs:
    inp = TriageInput(
        age=float(row[0]), cc_breathingdifficulty=float(row[1]), gender=float(row[2]),
        triage_vital_hr=float(row[3]), triage_vital_sbp=float(row[4]), triage_vital_rr=float(row[5]),
        triage_vital_o2=float(row[6]), pulse_min=float(row[7]), resp_min=float(row[8]),
        spo2_min=float(row[9]), sbp_min=float(row[10]), pulse_max=float(row[11]),
        resp_max=float(row[12]), spo2_max=float(row[13]), sbp_max=float(row[14])
    )
    res = c_lib.predict_triage(ctypes.byref(inp))
    c_preds.append(res.predicted_esi)
    c_probs.append(list(res.probs))
c_preds = np.array(c_preds)
c_probs = np.array(c_probs)
# Calculate Native Python/R Model Predictions on the same test set
# Native Python Meta-Learner Predictions
b_l1  = lgb.Booster(model_file=os.path.join(deploy_dir, 'l1_booster.txt'))
b_l2  = lgb.Booster(model_file=os.path.join(deploy_dir, 'l2_booster.txt'))
b_l3a = lgb.Booster(model_file=os.path.join(deploy_dir, 'l3a_booster.txt'))
b_l3b = lgb.Booster(model_file=os.path.join(deploy_dir, 'l3b_booster.txt'))
def compute_py_pipeline_probs(raw_mat):
    # Build 38 features in Python
    N = len(raw_mat)
    X38 = np.zeros((N, 38))
    X38[:, :15] = raw_mat
    t_o2 = raw_mat[:, 6]; t_rr = raw_mat[:, 5]; t_sbp = raw_mat[:, 4]; t_hr = raw_mat[:, 3]
    pulse_max = raw_mat[:, 11]; pulse_min = raw_mat[:, 7]
    resp_max  = raw_mat[:, 12]; resp_min  = raw_mat[:, 8]
    spo2_max  = raw_mat[:, 13]; spo2_min  = raw_mat[:, 9]
    sbp_max   = raw_mat[:, 14]; sbp_min   = raw_mat[:, 10]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    X38[:, 15] = (t_o2 < 90).astype(float)
    X38[:, 16] = ((t_o2 > 90) & (t_o2 < 94)).astype(float)
    X38[:, 17] = (t_rr < 10).astype(float)
    X38[:, 18] = (t_rr > 30).astype(float)
    X38[:, 19] = (t_sbp <= 90).astype(float)
    X38[:, 20] = (t_sbp > 220).astype(float)
    X38[:, 21] = (t_hr < 40).astype(float)
    X38[:, 22] = ((t_hr > 40) & (t_hr < 60)).astype(float)
    X38[:, 23] = (t_hr > 150).astype(float)
    X38[:, 24] = ((t_hr > 100) & (t_hr < 150)).astype(float)
    X38[:, 25] = hr_rng; X38[:, 26] = rr_rng; X38[:, 27] = spo2_rng; X38[:, 28] = sbp_rng
    X38[:, 29] = t_hr / np.where(t_sbp==0, 1, t_sbp)
    X38[:, 30] = t_hr - hr_rng
    X38[:, 31] = t_sbp - sbp_rng
    X38[:, 32] = t_rr - rr_rng
    X38[:, 33] = t_o2 - spo2_rng
    X38[:, 34] = t_o2 / np.where(t_rr==0, 1, t_rr)
    X38[:, 35] = spo2_rng / np.where(spo2_max==0, 1, spo2_max)
    X38[:, 36] = hr_rng / (t_hr + 1.0)
    X38[:, 37] = (t_rr / np.where(t_o2==0, 1, t_o2)) * 100.0
    
    # Scale continuous
    for idx, f_name in enumerate(feature_names):
        if f_name in scaler_cols:
            X38[:, idx] = (X38[:, idx] - scaler_means[f_name]) / scaler_sds[f_name]
            
    p1  = b_l1.predict(X38)
    p2  = b_l2.predict(X38)
    p3a = b_l3a.predict(X38)
    p3b = b_l3b.predict(X38)
    
    base_p = np.zeros((N, 5))
    base_p[:, 0] = p1
    base_p[:, 1] = (1 - p1) * p2 * p3a
    base_p[:, 2] = (1 - p1) * p2 * (1 - p3a)
    base_p[:, 3] = (1 - p1) * (1 - p2) * p3b
    base_p[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    
    return meta_logreg.predict_proba(base_p)
py_probs = compute_py_pipeline_probs(test_inputs)
py_preds = np.argmax(py_probs, axis=1) + 1
# Calculate Discrepancy & Dropoff
max_prob_diff = np.max(np.abs(c_probs - py_probs))
mismatches    = np.sum(c_preds != py_preds)
mismatch_pct  = (mismatches / len(y_test)) * 100.0
print("========================================================================")
print("   PURE C TRANSPILATION VERIFICATION & DISCREPANCY REPORT")
print("========================================================================")
print(f"  Total Test Samples Analyzed : {len(y_test)}")
print(f"  Max Probability Difference  : {max_prob_diff:.8f}")
print(f"  Prediction Class Mismatches : {mismatches} ({mismatch_pct:.2f}%)")
print("========================================================================\n")
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)
df_py = get_per_class_breakdown(y_test, py_preds, py_probs, 'Native_Python_OOF_Logistic_Pipeline')
df_c  = get_per_class_breakdown(y_test, c_preds,  c_probs,  'Transpiled_Pure_C_Pipeline')
comp_report_df = pd.concat([df_py, df_c], ignore_index=True)
print("========================================================================================")
print("   HOLDOUT TEST SET PER-CLASS COMPARISON: NATIVE PYTHON VS TRANSPILED PURE C")
print("========================================================================================")
print(comp_report_df.to_string(index=False))
print("========================================================================================\n")
reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)
comp_report_df.to_csv(os.path.join(reports_dir, 'c_transpilation_performance_dropoff_report.csv'), index=False)
print(f"Comparative Performance Dropoff Report saved to: {os.path.join(reports_dir, 'c_transpilation_performance_dropoff_report.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 6: Plot Performance Comparison Bar Chart (Native Python vs Pure C)
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
esi_classes_df = comp_report_df[comp_report_df['Class'] != 'Macro_Average']
df_melted = pd.melt(esi_classes_df, id_vars=['Pipeline', 'Class'], value_vars=['Recall', 'Specificity', 'ROC_AUC'], var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_melted, x='Class', y='Score', hue='Pipeline', palette=['#1f77b4', '#2ca02c'])
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=8, xytext=(0, 2),
                    textcoords='offset points')
plt.title('Performance Comparison: Native Python/R Pipeline vs Transpiled Pure C Shared Library', fontsize=12, fontweight='bold', pad=15)
plt.ylim(0, 1.15)
plt.ylabel('Score', fontsize=11)
plt.xlabel('ESI Triage Level', fontsize=11)
plt.legend(title='Runtime Environment', loc='upper right')
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'c_transpilation_vs_python_comparison.png'), dpi=300)
plt.show()
print(f"Comparison Plot saved to {os.path.join(plots_dir, 'c_transpilation_vs_python_comparison.png')}")